# Judge behavior runs with three OpenRouter models

This notebook loads existing `runs/behavior_*/behavior_results.csv` files and re-labels each `(statement, generation)` pair with three LLM judges via OpenRouter.

It does not overwrite the original `behavior_results.csv` files. For each behavior run it writes:

- `judge_3model_progress.csv` after each judge model finishes, so the run is resumable.
- `judge_3model_results.csv` with per-model labels/reasons and a simple majority label.

Before running, set `OPENROUTER_API_KEY` in the notebook environment.

In [ ]:
import os
import re
import sys
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "latent_alignment").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from latent_alignment.judge import DEFAULT_JUDGE_MODEL, judge_generations

ROOT

## 1. Configuration

Edit `JUDGE_MODELS` if you want a different three-model panel. The default list uses OpenRouter model IDs.

In [ ]:
RUNS_DIR = ROOT / "runs"
BEHAVIOR_GLOB = "behavior_*/behavior_results.csv"

JUDGE_MODELS = [
    "deepseek/deepseek-v4-flash",
    "openai/gpt-oss-120b",
    "qwen/qwen3.7-plus",
]

SAMPLE_N = None      # set to an int for a cheap smoke test
MAX_WORKERS = 8      # lower this if OpenRouter rate-limits the run
OVERWRITE = False    # False resumes existing judge_3model_progress.csv files
KEEP_RAW = False     # True stores raw judge JSON/text; larger CSVs

PROGRESS_NAME = "judge_3model_progress.csv"
RESULTS_NAME = "judge_3model_results.csv"

if len(JUDGE_MODELS) != 3:
    raise ValueError("Set exactly three judge models in JUDGE_MODELS")
if len(set(JUDGE_MODELS)) != len(JUDGE_MODELS):
    raise ValueError("JUDGE_MODELS should not contain duplicates")
if not os.environ.get("OPENROUTER_API_KEY"):
    raise RuntimeError("Set OPENROUTER_API_KEY before running OpenRouter judging")

JUDGE_MODELS

## 2. Find behavior datasets

In [ ]:
behavior_files = sorted(RUNS_DIR.glob(BEHAVIOR_GLOB))
if not behavior_files:
    raise FileNotFoundError(f"No files matched {RUNS_DIR / BEHAVIOR_GLOB}")

pd.DataFrame(
    {
        "run": [p.parent.name for p in behavior_files],
        "path": [str(p.relative_to(ROOT)) for p in behavior_files],
    }
)

## 3. Helpers

In [ ]:
REQUIRED_COLUMNS = {"statement", "generation"}


def model_slug(model: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", model.lower()).strip("_")


def judge_columns(model: str) -> tuple[str, str, str]:
    prefix = f"judge_{model_slug(model)}"
    return f"{prefix}_label", f"{prefix}_reason", f"{prefix}_raw"


def load_behavior(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    if SAMPLE_N is not None:
        df = df.head(SAMPLE_N).copy()
    return df


def resume_or_fresh(source_path: Path) -> pd.DataFrame:
    df = load_behavior(source_path)
    progress_path = source_path.parent / PROGRESS_NAME

    if progress_path.exists() and not OVERWRITE:
        progress = pd.read_csv(progress_path)
        if len(progress) == len(df):
            for col in progress.columns:
                if col.startswith("judge_") and col not in df.columns:
                    df[col] = progress[col]
        else:
            print(
                f"Ignoring {progress_path}: row count {len(progress)} != current {len(df)}"
            )

    return df


def missing_judgments(df: pd.DataFrame, label_col: str) -> pd.Series:
    if label_col not in df.columns:
        return pd.Series(True, index=df.index)
    labels = df[label_col].astype("string")
    return labels.isna() | labels.str.strip().eq("")


def majority_label(row: pd.Series):
    counts = row.dropna().astype(str).value_counts()
    if counts.empty:
        return pd.NA
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return "tie"
    return counts.index[0]


def add_consensus_columns(df: pd.DataFrame) -> pd.DataFrame:
    label_cols = [judge_columns(model)[0] for model in JUDGE_MODELS]
    present = [col for col in label_cols if col in df.columns]
    if len(present) != len(label_cols):
        return df

    labels = df[label_cols].astype("string")
    df["judge_3model_label"] = labels.apply(majority_label, axis=1)
    df["judge_3model_all_agree"] = labels.nunique(axis=1) == 1
    df["judge_3model_harmful_votes"] = labels.eq("harmful").sum(axis=1)
    df["judge_3model_safe_votes"] = labels.eq("safe").sum(axis=1)
    df["judge_3model_gibberish_votes"] = labels.eq("gibberish").sum(axis=1)
    return df

## 4. Run the three-model judge panel

This cell can take a while and consumes OpenRouter credits. It writes progress after every `(dataset, judge_model)` block.

In [ ]:
written = []

for source_path in tqdm(behavior_files, desc="behavior datasets"):
    print(f"\n=== {source_path.parent.name} ===")
    df = resume_or_fresh(source_path)
    progress_path = source_path.parent / PROGRESS_NAME
    results_path = source_path.parent / RESULTS_NAME

    for model in JUDGE_MODELS:
        label_col, reason_col, raw_col = judge_columns(model)
        todo_mask = missing_judgments(df, label_col)
        todo_idx = df.index[todo_mask].tolist()

        if not todo_idx:
            print(f"{model}: already complete")
            continue

        print(f"{model}: judging {len(todo_idx)} rows")
        verdicts = judge_generations(
            df.loc[todo_idx, "statement"].fillna("").astype(str).tolist(),
            df.loc[todo_idx, "generation"].fillna("").astype(str).tolist(),
            model=model,
            max_workers=MAX_WORKERS,
        )

        df.loc[todo_idx, label_col] = [v["label"] for v in verdicts]
        df.loc[todo_idx, reason_col] = [v["reason"] for v in verdicts]
        if KEEP_RAW:
            df.loc[todo_idx, raw_col] = [v.get("raw", "") for v in verdicts]

        add_consensus_columns(df)
        df.to_csv(progress_path, index=False)
        print(f"wrote {progress_path.relative_to(ROOT)}")

    add_consensus_columns(df)
    df.to_csv(results_path, index=False)
    written.append(results_path)
    print(f"wrote {results_path.relative_to(ROOT)}")

written

## 5. Summary

In [ ]:
summary_rows = []
for path in written:
    df = pd.read_csv(path)
    label_cols = [judge_columns(model)[0] for model in JUDGE_MODELS]

    row = {
        "run": path.parent.name,
        "n": len(df),
        "all_agree_rate": df["judge_3model_all_agree"].mean(),
        "majority_harmful_rate": (df["judge_3model_label"] == "harmful").mean(),
        "majority_safe_rate": (df["judge_3model_label"] == "safe").mean(),
        "majority_gibberish_rate": (df["judge_3model_label"] == "gibberish").mean(),
    }
    for col in label_cols:
        row[f"{col}_harmful_rate"] = (df[col] == "harmful").mean()
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary

In [ ]:
for path in written:
    df = pd.read_csv(path)
    print(f"\n=== {path.parent.name}: judge majority by original label ===")
    if "label" in df.columns:
        display(pd.crosstab(df["label"], df["judge_3model_label"], normalize="index"))
    else:
        display(df["judge_3model_label"].value_counts(normalize=True))